In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
PyTorch: 2.10.0+cu128
CUDA: 12.8


In [2]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
!pip install -q gdown

import gdown

DRIVE_ZIP_ID = "1zaMrF3lfM1a7AwkuUK1UkL8u0vEIag9W"
gdown.download(id=DRIVE_ZIP_ID, output="/kaggle/working/yolo_dataset.zip", quiet=False)

!unzip -q /kaggle/working/yolo_dataset.zip -d /kaggle/working/

Downloading...
From (original): https://drive.google.com/uc?id=1zaMrF3lfM1a7AwkuUK1UkL8u0vEIag9W
From (redirected): https://drive.google.com/uc?id=1zaMrF3lfM1a7AwkuUK1UkL8u0vEIag9W&confirm=t&uuid=18d5c86d-aa70-4f4f-b254-ba006e35a0c2
To: /kaggle/working/yolo_dataset.zip
100%|██████████| 368M/368M [00:10<00:00, 35.3MB/s] 


In [4]:
import os

DATASET = "/kaggle/working/yolo_dataset"

print("Dataset exists:", os.path.isdir(DATASET))
print("Contents:", os.listdir(DATASET))

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET, "images", split)
    label_dir = os.path.join(DATASET, "labels", split)

    print(
        split,
        "| images:", len(os.listdir(img_dir)),
        "| labels:", len(os.listdir(label_dir))
    )

Dataset exists: True
Contents: ['data.yaml', 'labels', 'images']
train | images: 8014 | labels: 8014
valid | images: 1136 | labels: 1136
test | images: 568 | labels: 568


In [5]:
import yaml

CLASS_NAMES = [
    "arc",
    "disconnector",
    "disconnector_open",
    "insulator",
    "spark",
    "switch",
    "switchgear",
    "transformer",
]

data_yaml = {
    "path": "/kaggle/working/yolo_dataset",
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

with open("/kaggle/working/yolo_dataset/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(open("/kaggle/working/yolo_dataset/data.yaml").read())

path: /kaggle/working/yolo_dataset
train: images/train
val: images/valid
test: images/test
nc: 8
names:
- arc
- disconnector
- disconnector_open
- insulator
- spark
- switch
- switchgear
- transformer



In [7]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.9 MB/s eta 0:00:00


In [8]:
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics: 8.4.146
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4


In [9]:
import os
import yaml
from ultralytics import RTDETR

DATASET = "/kaggle/working/yolo_dataset"
DATA_YAML = f"{DATASET}/data.yaml"

with open(DATA_YAML, "r") as f:
    data = yaml.safe_load(f)

print("Classes:", data["names"])
print("Number of classes:", data["nc"])

for split in ["train", "val", "test"]:
    path = os.path.join(DATASET, data[split])
    print(f"{split}: {path} | exists:", os.path.isdir(path))

model = RTDETR("rtdetr-l.pt")
print("RT-DETR-L loaded successfully.")
print("Parameters:", sum(p.numel() for p in model.model.parameters()))

Classes: ['arc', 'disconnector', 'disconnector_open', 'insulator', 'spark', 'switch', 'switchgear', 'transformer']
Number of classes: 8
train: /kaggle/working/yolo_dataset/images/train | exists: True
val: /kaggle/working/yolo_dataset/images/valid | exists: True
test: /kaggle/working/yolo_dataset/images/test | exists: True
RT-DETR-L loaded successfully.
Parameters: 32970476


In [10]:
import os
from ultralytics import RTDETR

CKPT_DIR = "/kaggle/working/rap_runs"
LAST_CKPT = os.path.join(CKPT_DIR, "train", "weights", "last.pt")

if os.path.exists(LAST_CKPT):
    print("Checkpoint found — resuming:", LAST_CKPT)
    model = RTDETR(LAST_CKPT)
    model.train(resume=True)
else:
    print("No checkpoint found — starting fresh RT-DETR-L training.")
    model = RTDETR("rtdetr-l.pt")
    model.train(
        data="/kaggle/working/yolo_dataset/data.yaml",
        epochs=30,
        imgsz=640,
        batch=8,
        device=0,
        project=CKPT_DIR,
        name="train",
        save_period=1,
        deterministic=False,
        seed=0,
    )

No checkpoint found — starting fresh RT-DETR-L training.
Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, m

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      7.05G     0.6327       0.73     0.3835         29        640: 100% ━━━━━━━━━━━━ 1002/1002 2.1it/s 7:590.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 71/71 3.4it/s 20.6s0.3ss
                   all       1136       3586      0.632      0.552      0.585      0.332

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/30      7.05G     0.6138     0.6917      0.359         34        640: 100% ━━━━━━━━━━━━ 1002/1002 2.1it/s 7:570.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 71/71 3.5it/s 20.5s0.3ss
                   all       1136       3586      0.615      0.617      0.604      0.346

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       4/30       7.1G     0.5874     0.6763     0.3448         22        640: 100% ━━━━━━━━━━━━ 1002/1002 2.1it/s 7:560.5s
                 Class    

In [11]:
from ultralytics import RTDETR

model = RTDETR("/kaggle/working/rap_runs/train/weights/best.pt")

metrics = model.val(
    data="/kaggle/working/yolo_dataset/data.yaml",
    split="test",
    plots=True,
)

print("TEST mAP50:", metrics.box.map50)
print("TEST mAP50-95:", metrics.box.map)

Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 25.5±31.4 MB/s, size: 39.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/yolo_dataset/labels/test... 568 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 568/568 869.6it/s 0.7s0.1s
val: New cache created: /kaggle/working/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 1.7it/s 21.1s0.6ss
                   all        568       1934      0.916      0.896      0.933      0.669
                   arc         76        119      0.918      0.924      0.958      0.567
          disconnector         65        171      0.918      0.922   

In [12]:
!cp /kaggle/working/rap_runs/train/weights/best.pt /kaggle/working/best.pt

In [13]:
from IPython.display import FileLink
FileLink("/kaggle/working/best.pt")

/kaggle/working/best.pt